In [ ]:
import torch
from torch import nn

In [5]:
class LSTM_CELL(nn.Module):
    def __init__(self,input_size,hidden_size):
        super().__init__()
        self.forget_gate=nn.Linear(input_size+hidden_size,hidden_size)
        self.input_gate=nn.Linear(input_size+hidden_size,hidden_size)
        self.output_gate=nn.Linear(input_size+hidden_size,hidden_size)
        self.candidate_gate=nn.Linear(input_size+hidden_size,hidden_size)
    def forward(self,x,previous_hidden,previous_cell):
        combined=torch.cat([x,previous_hidden],dim=1)
        input_gate=torch.sigmoid(self.input_gate(combined))
        output_gate=torch.sigmoid(self.output_gate(combined))
        forget_gate=torch.sigmoid(self.forget_gate(combined))
        candidate_gate=torch.tanh(self.candidate_gate(combined))
        new_cell=forget_gate*previous_cell+input_gate*candidate_gate
        new_hidden=output_gate*torch.tanh(new_cell)
        return new_hidden,new_cell

In [6]:
batch_size=16
input_size=32
hidden_size=64
sequence_length=5
x=torch.randn(batch_size,sequence_length,input_size)
h=torch.zeros(batch_size,hidden_size)
c=torch.zeros(batch_size,hidden_size)

layer=[]
model=LSTM_CELL(input_size,hidden_size)
for t in range(sequence_length):
    current=x[:,t,:]
    h,c=model(current,h,c)
    layer.append(h.unsqueeze(1))
output=torch.cat(layer,dim=1)
print(f"Output shape: {output.shape}")

Output shape: torch.Size([16, 5, 64])




---

## 1. Forget Gate

The forget gate decides how much information from the previous cell state should be retained.

$$
f_t = \sigma(W_f[h_{t-1}, x_t] + b_f)
$$

The sigmoid function produces values between 0 and 1.

* A value close to **0** means: forget this information.
* A value close to **1** means: keep this information.

---

## 2. Input Gate

The input gate decides how much new information should be stored.

$$
i_t = \sigma(W_i[h_{t-1}, x_t] + b_i)
$$


$$
\tilde{c}_t = \tanh(W_c[h_{t-1}, x_t] + b_c)
$$


---

## 3. Cell State Update


$$
c_t = f_t \odot c_{t-1} + i_t \odot \tilde{c}_t
$$

Where $\odot$ represents **element-wise multiplication**.


---

## 4. Output Gate

$$
o_t = \sigma(W_o[h_{t-1}, x_t] + b_o)
$$

The new hidden state is calculated as:

$$
h_t = o_t \odot \tanh(c_t)
$$

---

## Important Symbols
<center>

| Symbol    | Meaning                                |
| --------- | -------------------------------------- |
| $x_t$     | Current input                          |
| $h_{t-1}$ | Previous hidden state                  |
| $h_t$     | New hidden state                       |
| $c_{t-1}$ | Previous cell state                    |
| $c_t$     | New cell state                         |
| $\sigma$  | Sigmoid activation function            |
| $\tanh$   | Hyperbolic tangent activation function |
| $\odot$   | Element-wise multiplication            |
| $W$       | Learnable weights                      |
| $b$       | Learnable biases                       |
<center>